# **5G Deployment Example With R2lab:**

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UDM, UDR, UPF
- OAI split RAN: CU-CP, CU-UP and 2 DUs: `du01` with AW2S Jaguar in R2Lab and `du02` with AW2S Panther in R2Lab
- OAI FlexRIC
- COTS UEs: `qhat01`, `qhat02` and `qhat03` in R2Lab

First, we install the required dependencies:

## Install Dependencies:

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal serevers:

In [ ]:
! pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
! pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
! slices auth login

Install the Post-5G orchestrator:

In [ ]:
! pip install ../../ --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple

Install the Ansible dependencies:

In [ ]:
! ansible-galaxy install -r ../../ansible/requirements.yaml

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

The Post-5G orchestrator keeps the logical network description separate from the deployment-specific choices and evolving deployment state, so the same experiment can be adapted to different implementations or testbeds.

To achieve that, it uses two separate models:
- The Post-5G metamodel describing the entire network and its relationships using in pure 3GPP-standard way
- The deployment model describing implementation choices, target nodes, provider settings, and operational state

To deploy, you invoke lifecycle managers for the work you need. For example, the `ProvisioningManagement` class reserves or creates resources, `KubernetesManagement` prepares nodes and forms a cluster, `CoreManagement`, `RANManagement`, `RICManagement`, and `UEManagement` prepare and install workloads on their respective 5G components.

These stages are composable management operations, not necessarily one automatic "deploy everything" command.

Each stage exposes its particular set of actions that run the corresponding Ansible roles and Helm actions, often concurrently where possible.

In [35]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  jaguar:
    type: aw2s
    model: jaguar
    numberOfTxPorts: 2
    numberOfRxPorts: 2
    txAttenuationDB: 17

  panther:
    type: aw2s
    model: panther
    numberOfTxPorts: 2
    numberOfRxPorts: 2
    txAttenuationDB: 17

ran:
  components:
    cucp:
      type: cu_cp
      config:
        ric: flexric

    cuup:
      type: cu_up
      config:
        cu_cp: cucp
        ric: flexric
  
    du01:
      type: du
      config:
        radioConfiguration:
          nrBand: 78
          channelBandwidthMhz: 50
          pointAFrequencyArfcn: 626748
          ssbArfcn: 627360
          ssbPbchBlockPowerDbm: -10
          tddConfiguration:
            periodicityMilliseconds: 2.5
            nrofDownlinkSlots: 2
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 2
            nrofUplinkSymbols: 4
        gnbDuId: 1
        nrCellId: 1
        physicalCellId: 1
        ru: jaguar
        cu_cp: cucp
        cu_up: cuup
        ric: flexric

    du02:
      type: du
      config:
        radioConfiguration:
          nrBand: 78
          channelBandwidthMhz: 50
          pointAFrequencyArfcn: 626748
          ssbArfcn: 627360
          ssbPbchBlockPowerDbm: -10
          tddConfiguration:
            periodicityMilliseconds: 2.5
            nrofDownlinkSlots: 2
            nrofDownlinkSymbols: 6
            nrofUplinkSlots: 2
            nrofUplinkSymbols: 4
        gnbDuId: 2
        nrCellId: 2
        physicalCellId: 2
        ru: panther
        cu_cp: cucp
        cu_up: cuup
        ric: flexric

ric:
  components:
    flexric:
      type: ric

ues:
  qhat01:
    type: cots
    config:
      supi: "001010000000006"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du01

  qhat02:
    type: cots
    config:
      supi: "001010000000007"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du01

  qhat03:
    type: cots
    config:
      supi: "001010000000008"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du01
"""
)

To see the full content of this Meta Model:

In [ ]:
print(metamodel.to_yaml(exclude_defaults=False))

Next, to get the deployment model, we need initiate a `DeploymentModel` object.

The easiest way to do so is by providing a YAML input that describes our deployment.

In this case, we are going to use `sopnode-f1` and `sopnode-f2` that are still currently on the legacy POS, and `sopnode-f4` on the new POS.

For the radio hardaware, we are going to use `jaguar` and `panther` as RUs in R2Lab, as well as `qhat01`, `qhat02` and `qhat03` ad UEs in R2Lab.

For that, it is necessary that you change the values in between `<...>` to contain your actual [R2Lab](https://r2lab.inria.fr/tuto-010-registration.md) credentials (your R2Lab slice name, email, and password).

In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    extra_path: /Users/zimabrou/Projects/5g_blueprint/venv/bin # path to slices and slices-pos executables

provisioners:
  pos:
    type: pos
    params:
      attempt_reservation: true

  legacy_pos:
    type: legacy_pos
    params:
      attempt_reservation: false
      central_pos_ssh_info:
        hostname: duckburg.net.cit.tum.de
        port: 10022
        user: <YOUR_SLICES_USERNAME>

  r2lab:
    type: r2lab
    params:
      r2lab_slice: <YOUR_R2LAB_SLICE>
      reservation_params:
        email: <YOUR_R2LAB_EMAIL>
        password: <YOUR_R2LAB_PASSWORD>

nodes:
  sopnode-f1:
    provisioner: legacy_pos
    provisioning_params:
      boot_params: >-
        biosdevname=0 net.ifnames=0 mitigations=off intel_iommu=on iommu=pt
        selinux=0 enforcing=0 isolcpus=managed_irq,16-63
        nohz_full=16-63 nohz=on rcu_nocbs=16-63
        kthread_cpus=0-4 irqaffinity=0-4 rcu_nocb_poll
        nosoftlockup intel_pstate=disable idle=poll skew_tick=1
        tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 nosmt
    roles:
      - ran

  sopnode-f2:
    provisioner: legacy_pos
    provisioning_params:
      boot_params: >-
        biosdevname=0 net.ifnames=0 mitigations=off intel_iommu=on iommu=pt
        selinux=0 enforcing=0 isolcpus=managed_irq,16-63
        nohz_full=16-63 nohz=on rcu_nocbs=16-63
        kthread_cpus=0-4 irqaffinity=0-4 rcu_nocb_poll
        nosoftlockup intel_pstate=disable idle=poll skew_tick=1
        tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 nosmt
    roles:
      - ran

  sopnode-f4:
    provisioner: pos
    roles:
      - core
      - ric

ran:
  implementation: oai
  components:
    cucp:
      type: cu_cp
      advanced_config:
        useAdditionalOptions: "--telnetsrv --telnetsrv.shrmod ci"

    cuup:
      type: cu_up

    du01:
      type: du
      multus:
        enabled: true
        interfaces:
          - name: "ru"
            host_interface: "r2lab_aw2s"
            ip_address: "192.168.236.102"
      advanced_config:
        ssb_PositionsInBurst_Bitmap: 1
        num_tp_cores: 4
        rxfh_core_id: 12
        txfh_core_id: 14
        tp_cores: [16,18,20,22]
        pusch_TargetSNRx10: 290
        pucch_TargetSNRx10: 200
        initialDLBWPlocationAndBandwidth: 36300
        initialDLBWPcontrolResourceSetZero: 11
        initialULBWPlocationAndBandwidth: 36300
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 15
        p0_NominalWithGrant: -76
        p0_nominal: -90
        useAdditionalOptions: "--thread-pool 17,19,21,23,25,27,29,31"
      node_name: sopnode-f1

    du02:
      type: du
      multus:
        enabled: true
        interfaces:
          - name: "ru"
            host_interface: "r2lab_aw2s"
            ip_address: "192.168.236.104"
      advanced_config:
        ssb_PositionsInBurst_Bitmap: 2
        num_tp_cores: 4
        rxfh_core_id: 12
        txfh_core_id: 14
        tp_cores: [16,18,20,22]
        pusch_TargetSNRx10: 290
        pucch_TargetSNRx10: 200
        initialDLBWPlocationAndBandwidth: 36300
        initialDLBWPcontrolResourceSetZero: 11
        initialULBWPlocationAndBandwidth: 36300
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 15
        p0_NominalWithGrant: -76
        p0_nominal: -90
        useAdditionalOptions: "--thread-pool 17,19,21,23,25,27,29,31"
      node_name: sopnode-f2

ric:
  implementation: oai

rus:
  jaguar:
    provisioner: r2lab

  panther:
    provisioner: r2lab

ues:
  qhat01:
    type: cots
    provisioner: r2lab

  qhat02:
    type: cots
    provisioner: r2lab

  qhat03:
    type: cots
    provisioner: r2lab
"""
)

> *IMPORTANT NOTE*: Make sure your R2Lab credentials are not shared publically!

By default, the metamodel and deployement model will use Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
and one 5G slice (`internet`) with one DNN (`internet`).

### The AW2S configurations are referenced from: https://github.com/sopnode/oai5g-rru/blob/main/rru/gnb-config-aw2s.yaml

## **1.**   Provisioning nodes:

In [3]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await prov.provision_nodes()

## **2.**   Kubernetes deployement:

In [5]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(prov)

In [6]:
await k8s.deploy()


PLAY [sopnode-f4] **************************************************************

PLAY [sopnode-f2] **************************************************************

PLAY [sopnode-f1] **************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'sopnode-f2'] ***********

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'sopnode-f4'] ***********

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'sopnode-f1'] ***********
[WARNING]: Host 'sopnode-f4' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [sopnode-f4]

TASK [k8s/prepare : Install base packages required for Kubernetes on 'sopnode-f4'] ***
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at 

## **3.**   5G core deployement:

In [7]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(k8s)

In [75]:
await core.deploy()


PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-ausf' chart] *****

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-pcf' chart] ******

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nssf' chart] *****

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

PLAY [sopnode

## **4.**   RIC deployement:

In [76]:
from orchestrator.deployment import RICManagement

ric = RICManagement(core)

In [79]:
await ric.deploy()


PLAY [sopnode-f1] **************************************************************

TASK [helm/install : Deploy rendered 'oai-flexric' Helm chart in namespace 'oai-ric'] ***
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
[WARNING]: Deprecation warnings can be disabled by setting `deprecation_warnings=False` in ansible.cfg.
[DEPRECATION WARNING]: The 'status.values' return value is deprecated and will be removed in version 8.0.0. Use 'status.release_values' instead. This feature will be removed from collection 'kubernetes.core' version 8.0.0.
changed: [sopnode-f1]

TASK [helm/install : Wait for 'oai-flexric' pod to become ready] ***************
[WARNING]: kubernetes<24.2.0 is not supported or tested. Some features ma

## **5.**   RAN nodes tuning:

In [80]:
from orchestrator.deployment import Miscellaneous

misc = Miscellaneous(ric)

Using the `Miscellaneous` management class, we will run a custom playbook to apply the necessary netplan configuration for `sopnode-f1` and `sopnode-f2` to reach the R2Lab network, as well as other CPU and NIC optimization needed for a healthy RAN deployment.

#### The tuning roles and playbook are taken directly from: https://github.com/sopnode/5g_ansible 

In [12]:
await misc.run_playbook(
    playbook="/Users/zimabrou/Projects/5g_blueprint/examples/r2lab/tuning/ansible/playbooks/tuning.yaml",
    roles_paths=["/Users/zimabrou/Projects/5g_blueprint/examples/r2lab/tuning/ansible/roles"],
)


PLAY [Setup RAN nodes] *********************************************************

TASK [Gathering Facts] *********************************************************
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [sopnode-f1]
[WARNING]: Host 'sopnode-f2' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [sopnode-f2]

TASK [tuning/netplan : Upload netplan config to remote] ************************
changed: [sopnode-f1]
changed: [sopnode-f2]

TASK [tuning/netplan : Genera

## **6.**   RAN deployement:

First, we are going to provision the remaining resources (RUs and UEs):

In [83]:
prov_radio = ProvisioningManagement(misc)
await prov_radio.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhos

### Then we are going to deploy the OAI RAN control-plane and the first DU (`du01` with `jaguar`).

In [84]:
from orchestrator.deployment import RANManagement

ran = RANManagement(prov_radio)

In [85]:
await ran.deploy(components=["cucp", "cuup"])


PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cuup' chart] *********

TASK [helm/prepare : Create Helm build directory for 'oai-cucp' chart] *********
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [sopnode-f1][WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [sopno

In [91]:
await ran.deploy(components=["du01"])


PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du01' chart] *********
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [sopnode-f1]

TASK [helm/prepare : Copy local 'oai-du' Helm chart directory for 'oai-du01'] ***
ok: [sopnode-f1]

TASK [helm/prepare : Render 'oai-du01' Helm Chart.yaml file from Jinja template] ***
ok: [sopnode-f1]

TASK [helm/prepare : Render templated Helm files for 'oai-du01'] ***************
ok: [sopnode-f1] => (item=values.yaml)
changed: [sopnode-f1] => (item=config.yaml)

PLAY RECAP *********************************************************************
sopnode-f1                 : ok=4    changed=1

Check logs of `du01`:

In [92]:
print(await ran.get_logs(components=["du01"], quiet=True))

/proc/sys/kernel/core_pattern=core
No configuration file /opt/oai-gnb-aw2s/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

# Active DUs
Active_gNBs:
  - du01
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - gNB_ID: 0x1
    gNB_DU_ID: 0x1
    gNB_name: du01
    tracking_area_code: 1
    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1
    ##### Physical parameters:
    min_rxtxtime: 6
    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 1
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 627360
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 626748
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz60, 3=kHz120
 

The DU is deployed and ready for connections.

## **7.**   Connecting UEs:

In [93]:
from orchestrator.deployment import UEManagement

ue = UEManagement(ran)

In [94]:
await ue.connect()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [Connect 'qhat01'] ********************************************************

TASK [Connect 'qhat02'] ********************************************************

PLAY [localhost] ***************************************************************

Check `du01` logs after UEs were connected:

In [95]:
print(await ran.get_logs(components=["du01"], quiet=True))

/proc/sys/kernel/core_pattern=core
No configuration file /opt/oai-gnb-aw2s/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

# Active DUs
Active_gNBs:
  - du01
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - gNB_ID: 0x1
    gNB_DU_ID: 0x1
    gNB_name: du01
    tracking_area_code: 1
    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1
    ##### Physical parameters:
    min_rxtxtime: 6
    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 1
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 627360
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 626748
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz60, 3=kHz120
 

The 3 UEs are connected to the 5G network.

Let's verify their connectivity by pinging the UPF:

In [96]:
ping = await prov_radio.run_ue_command(command=["ping", "-c", "6", "11.1.1.1", "-I", "wwan0"], quiet=True)

In [97]:
print(ping["qhat01"])

PING 11.1.1.1 (11.1.1.1) from 11.1.1.3 wwan0: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=41.6 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=59.8 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=67.9 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=76.0 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=95.4 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=104 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5008ms
rtt min/avg/max/mdev = 41.590/74.086/103.884/20.989 ms


In [98]:
print(ping["qhat02"])

PING 11.1.1.1 (11.1.1.1) from 11.1.1.2 wwan0: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=28.5 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=26.8 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=22.3 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=28.3 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=27.2 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=25.7 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5008ms
rtt min/avg/max/mdev = 22.338/26.485/28.504/2.068 ms


In [99]:
print(ping["qhat03"])

PING 11.1.1.1 (11.1.1.1) from 11.1.1.4 wwan0: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=28.0 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=27.2 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=25.7 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=28.8 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=26.7 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=29.7 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5009ms
rtt min/avg/max/mdev = 25.673/27.690/29.726/1.341 ms


## **7.**   Deploy the second DU (`du02` with `panther`):

In [100]:
ran = RANManagement(ue)

In [101]:
await ran.deploy(components=["du02"])


PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du02' chart] *********
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [sopnode-f1]

TASK [helm/prepare : Copy local 'oai-du' Helm chart directory for 'oai-du02'] ***
ok: [sopnode-f1]

TASK [helm/prepare : Render 'oai-du02' Helm Chart.yaml file from Jinja template] ***
ok: [sopnode-f1]

TASK [helm/prepare : Render templated Helm files for 'oai-du02'] ***************
changed: [sopnode-f1] => (item=values.yaml)
changed: [sopnode-f1] => (item=config.yaml)

PLAY RECAP *********************************************************************
sopnode-f1                 : ok=4    chan

#### Check `du01`'s and `du02`'s logs before the F1-handover attempt:

In [102]:
logs = await ran.get_logs(components=["du01", "du02"], quiet=True)

In [103]:
print(logs["du01"])

/proc/sys/kernel/core_pattern=core
No configuration file /opt/oai-gnb-aw2s/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

# Active DUs
Active_gNBs:
  - du01
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - gNB_ID: 0x1
    gNB_DU_ID: 0x1
    gNB_name: du01
    tracking_area_code: 1
    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1
    ##### Physical parameters:
    min_rxtxtime: 6
    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 1
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 627360
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 626748
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz60, 3=kHz120
 

In [104]:
print(logs["du02"])

/proc/sys/kernel/core_pattern=core
No configuration file /opt/oai-gnb-aw2s/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

# Active DUs
Active_gNBs:
  - du02
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - gNB_ID: 0x1
    gNB_DU_ID: 0x2
    gNB_name: du02
    tracking_area_code: 1
    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 2
    ##### Physical parameters:
    min_rxtxtime: 6
    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 2
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 627360
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 626748
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz60, 3=kHz120
 

## **8.**   Trigger F1-Handover using OAI CU-CP's telnet server command:

Get one of the UE's CU UE ID (usally they are incremental integers starting from 1)

In [ ]:
print(await ran.get_logs(components=["cucp"], quiet=True))

Let's try to handover UE with CU UE ID 1 from `du01` to `du02`.

Reference: https://github.com/OPENAIRINTERFACE/openairinterface5g/blob/develop/doc/handover-tutorial.md

In [ ]:
print((await ran.run_command(
    command=[
        "bash",
        "-c",
        "'exec 3<>/dev/tcp/127.0.0.1/9090; "
        "printf \"ci trigger_f1_ho 1\\n\" >&3; "
        "IFS= read -r response <&3; "
        "printf \"%s\\n\" \"$response\"; "
        "exec 3<&-; "
        "exec 3>&-'",
    ],
    components=["cucp"], quiet=True))["cucp"])

RRC F1 handover triggered for UE 1



The handover has been triggered.

Let's check `du02`'s logs after the handover:

In [ ]:
print(await ran.get_logs(components=["du02"], quiet=True))

/proc/sys/kernel/core_pattern=core
No configuration file /opt/oai-gnb-aw2s/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

# Active DUs
Active_gNBs:
  - du02
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - gNB_ID: 0x1
    gNB_DU_ID: 0x2
    gNB_name: du02
    tracking_area_code: 1
    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 2
    ##### Physical parameters:
    min_rxtxtime: 6
    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 2
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 627360
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 626748
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz60, 3=kHz120
 

In this case, the F1-handover attempt did not work as expected. It will be debugged later. A possible cause of this failure is that both DUs were opperating at the exact same frequency and might have interfered with each other.

Let's disconnect the UEs for now:

In [ ]:
ue = UEManagement(ran)
await ue.disconnect(force=True)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [Disconnect 'qhat03'] *****************************************************

TASK [Disconnect 'qhat01'] *****************************************************

PLAY [localhost] ***************************************************************

And finally save this experiment's meta model and deployment model:

In [ ]:
ran.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

> *Note*: All your Ansible run logs are stored by action, component and execution timestamp in `~/post-5g/ansible-runner` by default, which you can of course change using the `runner_path` parameter in the `params.ansible` section of the deployment model.